In [1]:
!pip -q install azure-ai-documentintelligence azure-core
!pip install azure-storage-blob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.0/106.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.8/217.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 7.3 MB/s eta 0:00:00


In [2]:
"""
Batch OCR of all documents in an Azure Blob container
using Azure AI Document Intelligence (Prebuilt Read)
"""
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.storage.blob import ContainerClient
import numpy as np
import socket

print("running")
socket.gethostbyname("canadatest.cognitiveservices.azure.com")

# Secrets (Kaggle)
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

endpoint = user_secrets.get_secret("AZURE_DOCINT_ENDPOINT")
key = user_secrets.get_secret("AZURE_DOCINT_KEY")
container_sas_url = user_secrets.get_secret("AZURE_BLOB_CONTAINER_SAS_URL")

print("Endpoint loaded:", bool(endpoint))
print("Key loaded:", bool(key))
print("Container SAS loaded:", bool(container_sas_url))

# Clients
docint_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

container_client = ContainerClient.from_container_url(
    container_sas_url
)


# Helpers
def format_bounding_box(bounding_box):
    if not bounding_box:
        return "N/A"
    reshaped = np.array(bounding_box).reshape(-1, 2)
    return ", ".join(f"[{x}, {y}]" for x, y in reshaped)

# Analyze all blobs
print(container_client.list_blobs())
def analyze_all_documents():
    for blob in container_client.list_blobs():
        print(blob.name)

        # Skip non-doc files if needed
        if not blob.name.lower().endswith((".pdf", ".png", ".jpg", ".jpeg")):
            continue

        base_url, sas = container_sas_url.split("?", 1)
        blob_url = f"{base_url}/{blob.name}?{sas}"
        print(f"\n===== Processing: {blob.name} =====")

        poller = docint_client.begin_analyze_document(
            "prebuilt-read",
            AnalyzeDocumentRequest(url_source=blob_url)
        )

        result = poller.result()

        print("Document content preview:")
        print(result.content[:500], "...\n")

        for page in result.pages:
            print(f"--- Page {page.page_number} ---")
            print(
                f"Size: {page.width} x {page.height} ({page.unit})"
            )

            for line_idx, line in enumerate(page.lines):
                print(
                    f"Line {line_idx}: '{line.content}' "
                    f"Box: {format_bounding_box(line.polygon)}"
                )

            for word in page.words:
                print(
                    f"Word '{word.content}' "
                    f"(confidence: {word.confidence})"
                )


# Entry point
if __name__ == "__main__":
    analyze_all_documents()

running
Endpoint loaded: True
Key loaded: True
Container SAS loaded: True
<iterator object azure.core.paging.ItemPaged at 0x7de0fc37fc80>
Amazon/Amazon01.pdf

===== Processing: Amazon/Amazon01.pdf =====
Document content preview:
amazon.com
TEST INVOICE - FOR AUTOMATION PURPOSES ONLY
Details for Order #113-8456210-7723491 Order Placed: November 12, 2025 Order Total: $27.84
Items Ordered
Price
2 of: FlexAid Fabric Adhesive Bandages, Large 2x4, 40 Count Sold by: Amazon.com Condition: New
$15.98
1 of: CleanPrep Alcohol Swabs (Pack of 3 Boxes) Sold by: Amazon.com Condition: New
$11.86
Shipping Address: Jordan Matthews - Facilities Services 1500 Innovation Way Campus Box 4021 Normal, IL 61761
United States
Shipping Speed: FRE ...

--- Page 1 ---
Size: 8.5 x 11.0 (LengthUnit.INCH)
Line 0: 'amazon.com' Box: [0.5683, 0.6398], [1.7, 0.6398], [1.7, 0.7973], [0.5683, 0.8021]
Line 1: 'TEST INVOICE - FOR AUTOMATION PURPOSES ONLY' Box: [0.5683, 0.888], [3.8346, 0.888], [3.8346, 1.036], [0.5683, 1.040